# Phase 5, control set — VLM verification

Same script as the treated/held_out verification, pointed at `control_candidates/` instead. Expected acceptance rate is much higher (60-80%) because bound-mode candidates already have well-formatted captions.

## 1. Mount Drive and clone repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = "https://github.com/GabrielmAlves/dgm-2026.1.git"
BRANCH = "feature/generate_datasets"  
CANDIDATES_DRIVE = "/content/drive/MyDrive/binding-research/finetuning/control_candidates"

import os, subprocess
REPO_DIR = "/content/binding-research"
if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
%cd $REPO_DIR

assert os.path.exists(CANDIDATES_DRIVE), f'Upload candidates to {CANDIDATES_DRIVE} first'

## 2. Install deps

In [ ]:
!pip install -q uv
!uv pip install -e . --system

## 3. HF login

In [ ]:
from huggingface_hub import login
login()

## 4. Confirm GPU (A100 or L4)

In [ ]:
import torch
assert torch.cuda.is_available()
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {torch.cuda.get_device_name(0)} ({vram:.1f} GB)')
assert vram >= 20, 'Switch to A100 or L4'

## 5. Symlink and run verification

In [ ]:
!mkdir -p data/finetuning && ln -sfn $CANDIDATES_DRIVE data/finetuning/control_candidates
!ls data/finetuning/control_candidates/candidates_manifest.csv

!python experiments/exp5_verify_candidates.py \
    --candidates-root data/finetuning/control_candidates \
    --out-root /content/control_verified \
    --config configs/judge_default.yaml

## 6. Inspect

In [ ]:
import pandas as pd
appr = pd.read_csv('/content/control_verified/approved.csv')
rej  = pd.read_csv('/content/control_verified/rejected.csv')
print(f'Approved: {len(appr)}, Rejected: {len(rej)}')
print(f'Acceptance rate: {len(appr)/(len(appr)+len(rej)):.1%}')
print()
print('=== Per pair ===')
print(appr.groupby(['object','color']).size())
print()
print('=== Rejection reasons ===')
print(rej['rejection_reason'].value_counts())

## 7. Save to Drive

In [ ]:
import shutil
DRIVE_DEST = '/content/drive/MyDrive/binding-research/finetuning/control_verified'
shutil.copytree('/content/control_verified', DRIVE_DEST, dirs_exist_ok=True)
print(f'Saved to {DRIVE_DEST}')